# Advogato — Behavior Feature Computation

This notebook computes the **6 behavior features** for the Advogato trust network:

1. Activity
2. Popularity
3. Influence
4. Community Participation
5. Reciprocity
6. Reputation / Trustworthiness

**Before running:** download `hetrec` is NOT used here — Advogato comes from KONECT as `out.advogato` (inside `download.tsv.advogato.tar.bz2`, source: http://konect.cc/networks/advogato/). Place the extracted `out.advogato` file in the same folder as this notebook (or update `DATA_DIR` / `FILE_NAME` below).

**Important — file format:** KONECT `.tsv` edge files start with a few comment lines beginning with `%`, followed by rows of `source target weight` separated by spaces. The weight column holds Advogato's trust levels: **0.4 = Observer, 0.6 = Apprentice, 0.8 = Journeyer, 1.0 = Master**.


In [1]:
# If needed, install dependencies first (uncomment):
# !pip install pandas networkx python-louvain --quiet

import pandas as pd
import networkx as nx
import community as community_louvain  # from python-louvain package

DATA_DIR = "./"           # change this to the folder containing the file
FILE_NAME = "out.advogato"


## Step 1 — Load the raw trust edge list

In [2]:
edges = pd.read_csv(
    DATA_DIR + FILE_NAME,
    sep=" ",
    comment="%",
    names=["source", "target", "weight"]
)

print("Edges:", edges.shape)
edges.head()


Edges: (51127, 3)


,source,target,weight
0,1,1,0.8
1,1,2,1.0
2,1,3,1.0
3,1,4,1.0
4,1,5,1.0


## Step 2 — Build the directed, weighted trust graph

Unlike LastFM's friend network (undirected, unweighted), Advogato is **directed and weighted** —
`source -> target` with a `weight` representing the trust level source assigned to target.
This directionality matters: someone can trust you without you trusting them back.


In [3]:
DG = nx.from_pandas_edgelist(
    edges, source="source", target="target", edge_attr="weight",
    create_using=nx.DiGraph()
)
print("Nodes:", DG.number_of_nodes(), "| Edges:", DG.number_of_edges())


Nodes: 6539 | Edges: 51127


## Feature 1 — Activity

**Out-degree**: how many trust certifications this user *gave* to others.
Advogato has no separate content/listening file like LastFM, so activity here is entirely graph-derived — how actively a user participates in certifying others.


In [4]:
activity = pd.Series(dict(DG.out_degree()), name="activity_outdegree")
activity.sort_values(ascending=False).head(10)


157     786
597     544
232     378
9       188
1378    169
593     158
172     157
780     155
1222    153
1992    145
Name: activity_outdegree, dtype: int64

## Feature 2 — Popularity

**In-degree**: how many trust certifications this user *received* from others.


In [5]:
popularity = pd.Series(dict(DG.in_degree()), name="popularity_indegree")
popularity.sort_values(ascending=False).head(10)


46     722
30     518
328    370
126    335
286    254
438    228
719    223
329    213
22     191
739    189
Name: popularity_indegree, dtype: int64

## Feature 3 — Influence

- **PageRank** (weighted by trust level): how influential a user is, factoring in both how many people trust them and how trusted *those* people are
- **Betweenness Centrality**: how often this user acts as a bridge connecting otherwise disconnected parts of the trust network


In [6]:
pagerank    = pd.Series(nx.pagerank(DG, weight="weight"), name="pagerank")
betweenness = pd.Series(nx.betweenness_centrality(DG), name="betweenness")

influence = pd.concat([pagerank, betweenness], axis=1)
influence.sort_values("pagerank", ascending=False).head(10)


,pagerank,betweenness
719,0.018166,0.000000
46,0.010074,0.031253
30,0.006791,0.002083
328,0.004171,0.013508
126,0.003910,0.000000
286,0.002812,0.000000
353,0.002716,0.000074
1115,0.002681,0.000135
22,0.002576,0.003521
282,0.002388,0.001900


## Feature 4 — Community Participation

Louvain community detection requires an **undirected** graph, so we convert the trust graph first
(this loses direction but keeps the connection structure, which is standard practice for this step).


In [7]:
UG = DG.to_undirected()
partition = community_louvain.best_partition(UG)
community_col = pd.Series(partition, name="community_id")

print("Number of communities found:", community_col.nunique())
community_col.head(10)


Number of communities found: 1465


1       0
2       0
3       0
4       1
5       1
6       2
7       3
8       1
9       0
10    178
Name: community_id, dtype: int64

## Feature 5 — Reciprocity

Fraction of a user's outgoing trust links that are mutual (A trusts B **and** B trusts A).
In a trust network this is a meaningful signal on its own — one-directional trust (e.g. a newcomer certifying
a well-known developer who doesn't certify back) is very different from mutual trust between peers.


In [8]:
edge_set = set(zip(edges["source"], edges["target"]))

out_edges_by_user = {}
for u, v in zip(edges["source"], edges["target"]):
    out_edges_by_user.setdefault(u, []).append((u, v))

def fraction_mutual(u):
    out_e = out_edges_by_user.get(u, [])
    if not out_e:
        return 0.0
    mutual = sum(1 for (a, b) in out_e if (b, a) in edge_set)
    return mutual / len(out_e)

reciprocity = pd.Series(
    {u: fraction_mutual(u) for u in edges["source"].unique()},
    name="reciprocity"
)
reciprocity.sort_values(ascending=False).head(10)


47      1.0
6540    1.0
4833    1.0
4834    1.0
4376    1.0
4837    1.0
575     1.0
4840    1.0
4841    1.0
1163    1.0
Name: reciprocity, dtype: float64

## Feature 6 — Reputation / Trustworthiness

**Weighted in-degree**: sum of all incoming trust weights (0.4-1.0 per edge) a user has received.
This is the most direct trust score in the whole dataset — unlike LastFM where we needed a proxy (total plays),
Advogato's edge weights *are literally* trust certification levels, so this sum is a genuine, purpose-built trust metric.


In [9]:
reputation = edges.groupby("target")["weight"].sum().rename("trust_score_weighted_indegree")
reputation.sort_values(ascending=False).head(10)


target
46     720.2
30     515.2
328    367.0
126    332.8
286    250.0
719    222.0
329    211.0
22     190.0
739    187.0
438    177.8
Name: trust_score_weighted_indegree, dtype: float64

## Step 3 — Merge everything into one feature table


In [10]:
final_features = pd.concat(
    [activity, popularity, pagerank, betweenness, community_col, reciprocity, reputation],
    axis=1
)
final_features.index.name = "userID"

print("Final shape:", final_features.shape)
final_features.head(10)


Final shape: (6539, 7)


,activity_outdegree,popularity_indegree,pagerank,betweenness,community_id,reciprocity,trust_score_weighted_indegree
userID,,,,,,,
1,9,14,0.000327,0.000526,0,0.666667,12.4
2,3,8,0.000337,0.000205,0,1.000000,7.4
3,3,10,0.000340,0.000008,0,1.000000,9.4
4,23,30,0.000293,0.000161,1,0.391304,24.2
5,8,41,0.000720,0.000070,1,0.750000,40.6
6,2,16,0.000132,0.000008,2,1.000000,12.8
7,26,12,0.000091,0.000489,3,0.461538,8.4
8,11,10,0.000132,0.000101,1,0.272727,8.6
9,188,59,0.000425,0.006358,0,0.239362,47.0


## Step 4 — Save to CSV

In [11]:
final_features.to_csv("advogato_behavior_features.csv")
print("Saved: advogato_behavior_features.csv")


Saved: advogato_behavior_features.csv


## Sanity checks (recommended before using this in your report)

- Compare `activity_outdegree` / `popularity_indegree` against Gephi's In-Degree/Out-Degree columns for the same graph — should match exactly.
- Compare `trust_score_weighted_indegree` against Gephi's **Weighted In-Degree** statistic — should match exactly, since it's the same calculation.
- Run Gephi's built-in **Reciprocity** graph-level statistic and sanity-check it roughly matches the *average* of this notebook's per-user `reciprocity` column.
- Check for isolated users (activity=0 and popularity=0) — these exist in the graph edge list but have no in/out trust edges from this file alone shouldn't happen, but worth a quick `final_features[(final_features.activity_outdegree==0) & (final_features.popularity_indegree==0)]` check.


In [12]:
# Quick check: users with zero activity AND zero popularity (unlikely, but worth checking)
final_features[(final_features["activity_outdegree"] == 0) & (final_features["popularity_indegree"] == 0)]


,activity_outdegree,popularity_indegree,pagerank,betweenness,community_id,reciprocity,trust_score_weighted_indegree
userID,,,,,,,


In [14]:
print("Mean out-degree (notebook):", final_features["activity_outdegree"].mean())
print("Mean in-degree (notebook):", final_features["popularity_indegree"].mean())
print("Gephi's reported Avg Degree: 7.819")

Mean out-degree (notebook): 7.8187796299128305
Mean in-degree (notebook): 7.8187796299128305
Gephi's reported Avg Degree: 7.819
